In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")


In [ ]:

# Let's chat for 5 lines
for step in range(5):
    # encode the new user input, add the eos_token and return a tensor in Pytorch
    new_user_input_ids = tokenizer.encode(input(">> User:") + tokenizer.eos_token, return_tensors='pt')

    # append the new user input tokens to the chat history
    bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1) if step > 0 else new_user_input_ids

    # generated a response while limiting the total chat history to 1000 tokens,
    chat_history_ids = model.generate(bot_input_ids, max_length=1000, pad_token_id=tokenizer.eos_token_id)

    # pretty print last ouput tokens from bot
    print("DialoGPT: {}".format(tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)))


In [5]:
import pandas as pd

In [7]:
order_data = {
    "numero_pedido": ["12345", "67890", "11121", "22232"],
    "status": ["Shipped", "Processing", "Delivered", "Cancelled"]
}
df_order_data = pd.DataFrame(order_data)
df_order_data

,numero_pedido,status
0,12345,Shipped
1,67890,Processing
2,11121,Delivered
3,22232,Cancelled


In [8]:
def verify_order_status(order_number):
    try:
        status = df_order_data[df_order_data["numero_pedido"] == order_number]['status'].iloc[0]
        return f'The status of your order {order_number} is {status}.'
    except:
        return f'The order number {order_number} is not found. Please check the order number and try again.'

In [9]:
key_words_status = ["order", "order status", "status of my order", "check my order", "track my order", "order update"]

In [11]:
ids_historic_chat = None

while True:
    user_input = input("You: ")
    if(user_input.lower() in ["exit", "quit", "stop"]):
        print(f"Bot: Bye!")
        break
    if any(keyword in user_input.lower() for keyword in key_words_status):
        order_number = input('Could you please enter your order number?')
        response = verify_order_status(order_number)
    else:
        new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')
        if(ids_historic_chat is not None):
            bot_input_ids = torch.cat([ids_historic_chat, new_user_input_ids], dim=-1)
        else:
            bot_input_ids = new_user_input_ids
        ids_historic_chat = model.generate(
            bot_input_ids,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id
        )
        response = tokenizer.decode(ids_historic_chat[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
    print(f'Bot: {response}')


Bot: Hello ! :D
Bot: Hello ! :D
Bot: The status of your order 12345 is Shipped.
Bot: Bye!


In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import pandas as pd
import gradio as gr

tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")
def answer(input_usuario, ids_historic_chat):
  if any(keyword in input_usuario.lower() for keyword in key_words_status):
    return 'Could you please enter you order number?', ids_historic_chat
  else:
    new_user_input_ids = tokenizer.encode(input_usuario + tokenizer.eos_token, return_tensors='pt')

    if ids_historic_chat is not None:
      bot_input_ids = torch.cat([ids_historic_chat, new_user_input_ids ], dim=-1)

    else:
      bot_input_ids = new_user_input_ids

    ids_historic_chat = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(ids_historic_chat[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
    return response, ids_historic_chat


Loading weights: 100%|██████████| 293/293 [00:00<00:00, 24131.23it/s]


In [18]:
with gr.Blocks() as app:
  chatbot = gr.Chatbot()
  msg = gr.Textbox(placeholder='Type your message here...')

  state = gr.State(None)
  waiting_order_number = gr.State(False)

  def process_input(input_user, historic, ids_historic_chat, waiting_order_number):
    if(waiting_order_number):
        response = verify_order_status(input_user)
        waiting_order_number = False
    else:
        response, ids_historic_chat = answer(input_user, ids_historic_chat)
        if(response == 'Could you please enter you order number?'):
            waiting_order_number = True
    historic.append({"role": "user", "content": input_user})
    historic.append({"role": "assistant", "content": response})
    return historic, ids_historic_chat, waiting_order_number, ""

  msg.submit(
      process_input,
      [msg, chatbot, state, waiting_order_number],
      [chatbot, state, waiting_order_number, msg],
  )

  app.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
